In [1]:
import requests
import json
import re
import pandas as pd
import numpy as np

In [2]:
def get_uniprot(accession):
    '''
    define http_function to get the data from Uniprot API
    '''
    url = f"https://rest.uniprot.org/uniprotkb/{accession}"
    return requests.get(url, headers={"Accept": "application/json"})

In [3]:
def uniprot_parse_response(resp: dict):
    '''
    parse response from Uniprot and output
    organism, geneInfo, sequenceInfo, type

    do not forget to include error handling (e.g. for key errors)
    '''
    try:
        data = resp.json() if hasattr(resp, 'json') else resp
        
        accession = data.get('primaryAccession', 'Unknown')
        organism_data = data.get('organism', {})
        organism = organism_data.get('scientificName', organism_data) if isinstance(organism_data, dict) else organism_data
        
        output = {
            accession: {
                'organism': organism,
                'geneInfo': data.get('genes', []),
                'sequenceInfo': data.get('sequence', {}),
                'type': 'protein'
            }
        }
        return output
    except Exception as e:
        return {"error": str(e)}

In [ ]:
def get_ensembl(id):
  '''
  define http_function to get the data from Ensembl API
  '''
  url = f"https://rest.ensembl.org/lookup/id/{id}?content-type=application/json"
  if str(id).startswith('MGP_'):
      strain = str(id).split('_')[1].lower() # Extracts 'aj'
      url += f"&species=mus_musculus_{strain}&db_type=core"
  return requests.get(url)

In [18]:
def ensembl_parse_response(resp):
  '''
  parse Ensembl response and output
  object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source
  '''
  try:
    data = resp.json() if hasattr(resp, 'json') else resp
        
    ensembl_id = data.get('id', 'Unknown')
    keys_to_extract = [
        'object_type', 'assembly_name', 'species', 'db_type', 'biotype', 
        'display_name', 'id', 'description', 'canonical_transcript', 'source'
    ]
    output = {
        ensembl_id: {k: data.get(k) for k in keys_to_extract if k in data}
    }
    return output
  except Exception as e:
    return {"error": str(e)}

In [ ]:
def main(ids: list):
    '''
    Function that iterates over all the provided IDs and parses them into dict,
    transforms into pandas.DataFrame, and return it
    '''
    result_dict = {}

    uniprot_regex = re.compile(r'^[O,P,Q][0-9][A-Z,0-9]{3}[0-9]$|^[A-N,R-Z][0-9]([A-Z][A-Z,0-9]{2}[0-9]){1,2}$')

    for obj_id in ids:
        if str(obj_id).startswith('ENS') or str(obj_id).startswith('MGP_'):
            resp = get_ensembl(obj_id)
            if resp.status_code == 200:
                parsed = ensembl_parse_response(resp)
                result_dict.update(parsed)
            else:
                try:
                    result_dict[obj_id] = {'error': resp.json().get('error', 'Error occurred')}
                except:
                    result_dict[obj_id] = {'error': f"HTTP Error {resp.status_code}"}
                    
        elif uniprot_regex.match(str(obj_id)):
            resp = get_uniprot(obj_id)
            if resp.status_code == 200:
                parsed = uniprot_parse_response(resp)
                result_dict.update(parsed)
            else:
                try:
                    result_dict[obj_id] = {'error': resp.json().get('messages', ['Error occurred'])[0]}
                except:
                    result_dict[obj_id] = {'error': f"HTTP Error {resp.status_code}"}
                    
        else:
            result_dict[obj_id] = {'error': 'error:unknown database'}

    return pd.DataFrame.from_dict(result_dict, orient='index')

The code below is just test_data.txt self-checking:

In [7]:
get_uniprot('P11473')

<Response [200]>

In [8]:
get_uniprot('helloworld')

<Response [400]>

In [9]:
get_uniprot('helloworld').json()

{'url': 'http://rest.uniprot.org/uniprotkb/helloworld',
 'messages': ["The 'accession' value has invalid format. It should be a valid UniProtKB accession"]}

In [10]:
uniprot_parse_response(get_uniprot('P11473'))

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'protein'}}

In [11]:
get_ensembl('ENSMUSG00000041147')

<Response [200]>

In [12]:
get_ensembl('helloworld')

<Response [400]>

In [13]:
get_ensembl('helloworld').json()

{'error': "ID 'helloworld' not found"}

In [14]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))

{'ENSMUSG00000041147': {'object_type': 'Gene',
  'assembly_name': 'GRCm39',
  'species': 'mus_musculus',
  'db_type': 'core',
  'biotype': 'protein_coding',
  'display_name': 'Brca2',
  'id': 'ENSMUSG00000041147',
  'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
  'canonical_transcript': 'ENSMUST00000044620.11',
  'source': 'ensembl_havana'}}

In [27]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618', 'MGP_AJ_G0023128'])

,organism,geneInfo,sequenceInfo,type,error,object_type,assembly_name,species,db_type,biotype,display_name,id,description,canonical_transcript,source
P11473,Homo sapiens,[{'geneName': {'evidences': [{'evidenceCode': ...,{'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q91XI3,Ictidomys tridecemlineatus,[{'geneName': {'value': 'INS'}}],{'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hello,NaN,NaN,NaN,NaN,error:unknown database,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MGP_AJ_G0023128,NaN,NaN,NaN,NaN,ID 'MGP_AJ_G0023128' not found,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSG00000157764,NaN,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRAF,ENSG00000157764,"B-Raf proto-oncogene, serine/threonine kinase ...",ENST00000646891.2,ensembl_havana
ENSG00000139618,NaN,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRCA2,ENSG00000139618,BRCA2 DNA repair associated [Source:HGNC Symbo...,ENST00000380152.8,ensembl_havana
